#Ingest circuits.csv file
1. Read the file using spark dataframe reader API
2. Add Metadata columns:
    - Source File
    - Ingestion Timestamp
3. Write to bronze delta table


In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
source_file = f"{landing_folder_path}/circuits.csv"
table_name = f"{catalog_name}.{bronze_schema}.circuits"

In [0]:
table_name

#### Step 1 -  Read the CSV file using the dataframe reader API

In [0]:
from pyspark.sql.types import *
circuit_schema = StructType([
    StructField('circuitId', StringType(), True),
    StructField('url', StringType(), True),
    StructField('circuitName', StringType(), True),
    StructField('lat', DoubleType(), True),
    StructField('long', DoubleType(), True),
    StructField('locality', StringType(), True),
    StructField('country', StringType(), True)
])



In [0]:
circuits_df= (
    spark.read
    .format('csv')
    .option('header','true')
    #.option('inferSchema','true')
    .option('mode','FAILFAST')
    .schema(circuit_schema)
    .load(source_file)
    )

In [0]:
#circuits_df.show()
display(circuits_df)

#### Step 2 - Add Metadata columns:
  - Source File
  - Ingestion Timestamp

In [0]:
circuits_final_df = add_ingestion_metadata(circuits_df)

In [0]:
display(circuits_final_df)

#### Step 3 -  Write to bronze delta table

In [0]:
(
    circuits_final_df
    .write
    .format('delta')
    .mode('overwrite')
    .saveAsTable(table_name)
)

In [0]:
%sql
SELECT * FROM formula1.bronze.circuits;

In [0]:
display(spark.read.table(table_name))